In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 02 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 02.C1 — Manhattan distance

Calculate the sum of absolute coordinate differences for two equal-length vectors.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Absolute value removes direction; then add contributions from all selected coordinates.

</details>

In [3]:
def manhattan(x,y):
    return sum(abs(a-b) for a,b in zip(x,y))

In [4]:
assert manhattan([0,0],[3,4])==7
X=load_iris_data().iloc[:2,1:5].to_numpy()
assert np.isclose(manhattan(X[0],X[1]),np.abs(X[0]-X[1]).sum())
print("First two measured flowers, L1 distance:",manhattan(X[0],X[1]))

First two measured flowers, L1 distance: 0.6999999999999993


**Why this works and what it does not establish:** Different metrics express different notions of closeness. A seven-unit Manhattan distance between (0,0) and (3,4) does not contradict their five-unit Euclidean distance.

## 02.C2 — A Euclidean distance matrix

Return the square matrix D whose entry i,j is the Euclidean distance between observations i and j.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Broadcasting constructs one difference vector per pair; sum its squared coordinates, then take a square root.

</details>

In [5]:
def distances(X):
    X=np.asarray(X,dtype=float)
    return np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(axis=2))

In [6]:
D=distances([[0,0],[3,4],[3,0]])
assert np.allclose(D,[[0,5,3],[5,0,4],[3,4,0]])
X=load_iris_data().iloc[:8,1:5].to_numpy();D=distances(X)
assert np.allclose(D,D.T) and np.allclose(np.diag(D),0)
print("Real-data distance matrix shape:",D.shape)

Real-data distance matrix shape: (8, 8)


**Why this works and what it does not establish:** An n by n distance matrix is not the original n by d feature matrix. Its entries carry the units and scaling choices of the input coordinates.

## 02.C3 — Open neighborhoods versus graph thresholds

Given distances from a center, return indices strictly inside radius r. The open-ball definition uses <, not <=.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Use np.flatnonzero on a boolean comparison. Read the equality case before coding.

</details>

In [7]:
def open_ball_indices(distances_from_center,r):
    return set(np.flatnonzero(np.asarray(distances_from_center)<r).tolist())

In [8]:
assert open_ball_indices([0,1,2],1)=={0}
X=load_iris_data().iloc[:10,1:5].to_numpy()
d=np.linalg.norm(X-X[0],axis=1)
print("Sample IDs in radius 0.5 (restricted to first 10 rows):",sorted(open_ball_indices(d,.5)))

Sample IDs in radius 0.5 (restricted to first 10 rows): [0, 4, 7, 9]


**Why this works and what it does not establish:** Topological open balls and our closed threshold Rips edges use different endpoint conventions. Tiny differences at equal values can change hand-derived birth and death endpoints.

## 02.C4 — Validate a finite topology

For a finite universe U and a finite collection of candidate open sets, check empty/full membership, containment, and closure under pairwise unions and intersections.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Convert sets to frozenset so a collection of them can itself be stored as a set.

</details>

In [9]:
def finite_topology(U,opens):
    U=frozenset(U);T=set(map(frozenset,opens))
    return (frozenset() in T and U in T and all(A<=U for A in T)
            and all(A|B in T and A&B in T for A in T for B in T))

In [10]:
assert finite_topology({0,1},[set(),{1},{0,1}])
assert not finite_topology({0,1},[set(),{1}])
assert not finite_topology({0,1,2},[set(),{0},{1},{0,1,2}])
print("Finite topology examples checked")

Finite topology examples checked


**Why this works and what it does not establish:** For a finite collection, binary union closure yields arbitrary union closure because only finitely many distinct members can contribute. This finite shortcut does not replace the arbitrary-union axiom for infinite spaces.

## 02.C5 — Find a neighbor without leaking labels

Return the nearest distinct row position to row i using only the given numeric features. Break ties with the smallest row position.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Exclude the center itself by setting its candidate distance to infinity.

</details>

In [11]:
def nearest_other(X,i):
    X=np.asarray(X,dtype=float)
    d=np.linalg.norm(X-X[i],axis=1)
    d[i]=np.inf
    return int(np.argmin(d))

In [12]:
assert nearest_other([[0],[1],[3]],1)==0
X=load_iris_data().iloc[:,1:5].to_numpy()
j=nearest_other(X,0)
assert j!=0
print("Nearest other row to Iris sample 0:",j)

Nearest other row to Iris sample 0: 17


**Why this works and what it does not establish:** Duplicate measured coordinates can occur at distinct row IDs; they may be valid zero-distance neighbors. That creates a pseudometric on observation identities, not an excuse to silently delete observations.